In [1]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"

In [ ]:
S0_2_R0_child = pd.read_csv("S0.2-R0_child_results.csv")
S0_2_R0_node = pd.read_csv("S0.2-R0_node_stats.csv")
S0_2_R0_parent = pd.read_csv("S0.2-R0_parent_results.csv")
S0_2_R0_static = pd.read_csv("S0.2-R0_static_info.csv")
S0_2_R0_unary = pd.read_csv("S0.2-R0_unary_results.csv")
# S0_2_R0_child.head()

S0_2_R1_child = pd.read_csv("S0.2-R1_child_results.csv")
S0_2_R1_node = pd.read_csv("S0.2-R1_node_stats.csv")
S0_2_R1_parent = pd.read_csv("S0.2-R1_parent_results.csv")
S0_2_R1_static = pd.read_csv("S0.2-R1_static_info.csv")
S0_2_R1_unary = pd.read_csv('S0.2-R1_unary_results.csv')

S0_2_R2_child = pd.read_csv("S0.2-R2_child_results.csv")
S0_2_R2_node = pd.read_csv("S0.2-R2_node_stats.csv")
S0_2_R2_parent = pd.read_csv("S0.2-R2_parent_results.csv")
S0_2_R2_static = pd.read_csv("S0.2-R2_static_info.csv")
S0_2_R2_unary = pd.read_csv("S0.2-R2_unary_results.csv")

S0_2_R3_child = pd.read_csv("S0.2-R3_child_results.csv")
S0_2_R3_node = pd.read_csv("S0.2-R3_node_stats.csv")
S0_2_R3_parent = pd.read_csv("S0.2-R3_parent_results.csv")
S0_2_R3_static = pd.read_csv("S0.2-R3_static_info.csv")
S0_2_R3_unary = pd.read_csv("S0.2-R3_unary_results.csv")


unary = pd.concat([S0_2_R0_unary, S0_2_R1_unary, S0_2_R2_unary, S0_2_R3_unary], ignore_index=True)
child = pd.concat([S0_2_R0_child, S0_2_R1_child, S0_2_R2_child, S0_2_R3_child], ignore_index=True)
node = pd.concat([S0_2_R0_node, S0_2_R1_node, S0_2_R2_node, S0_2_R3_node], ignore_index=True)
parent = pd.concat([S0_2_R0_parent, S0_2_R1_parent, S0_2_R2_parent, S0_2_R3_parent], ignore_index=True)


In [ ]:
def heat_map(results, static, prefix):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    fig = go.Figure(go.Histogram2d(
    histnorm="density",
    x = results.iloc[:, 1],
    y = results.iloc[:, 2] - results.iloc[:, 3],
    autobinx=False,
    xbins=dict(start=250, end=2250, size=50),
    autobiny=False,
    ybins=dict(start=-0.15, end=0.15, size=0.01)
    ))
    fig.update_layout(title=f"Sigma{sigma} Heatmap of Error Difference in {prefix} Nodes by Time", xaxis_title='Time', yaxis_title='Simplified Error - Extended Error')
    fig.show()


In [7]:
def error_difference_by_time_bin(results, static):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8
    
    results['time_bin'], bin_edges = pd.qcut(
        results.iloc[:, 1], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )
    
    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['time_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    
    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
    
        bin_data = results[results['time_bin'] == bin_interval]
        diff = bin_data.iloc[:, 2] - bin_data.iloc[:, 3]

        fig.add_trace(go.Histogram(
            x=diff, 
            name='Simplified - Extended',
            opacity=0.6, 
            marker_color='red',
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_layout(
        barmode='overlay', 
        title=f"S{sigma} R{rep} Error Difference (Equal-Count Bins)", 
        height=600
        # template="plotly_white"
    )
    fig.show()

In [13]:
def avg_error_difference_by_time_bin(results, static):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8
    
    results['time_bin'], bin_edges = pd.qcut(
        results.iloc[:, 1], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['time_bin'].unique())

    fig = go.Figure()

   #fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    array = np.zeros(8)
    count = 0

    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
        
        bin_data = results[results['time_bin'] == bin_interval]
        
        simp_errors = bin_data.iloc[:, 2]
        avg_simp_errors = np.mean(simp_errors)

        ets_errors = bin_data.iloc[:, 3]
        avg_ets_errors = np.mean(ets_errors)

        x = avg_simp_errors - avg_ets_errors
        array[count] = x
        count = count + 1

    fig.add_trace(go.Scatter(x=labels, y=array,  mode='markers'))

    fig.update_layout(title=f"S{sigma} R{rep} Average Error Difference by Time Bin", 
                        xaxis_title='Time Bins', yaxis_title='Average Simplified Error - Extended Error')
    # fig.update_yaxes(type='log')
    fig.show()
    
  


In [ ]:
def error_vs_added_span(results, static):
    fig = go.Figure()

    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]

    fig.add_trace(go.Scatter(x=results.iloc[:, 7], y = results.iloc[:, 3], mode='markers', name='Extended'))
    fig.add_trace(go.Scatter(x=results.iloc[:, 7], y = results.iloc[:, 2], mode='markers', name='Simplified'))

    fig.update_layout(title=f"S{sigma} R{rep} Simplified Vs Extended Error by Added Span", xaxis_title='Added Span', yaxis_title='Error')
    fig.show()



    # ancestor_df = pd.DataFrame({
    #  0   'node_id': target_node_ids,
    #  1   'node_time': target_node_times,
    #  2   'simp_error': simp_e,
    #  3   'ets_error': ets_e,
    #  4   'simp_span': target_simp_spans,
    #  5   'ets_span': target_ets_spans,
    #  6   'added_span': target_total_added_span,
    #  7   'wrongly_added_span': target_wrong_added_span,
    #  8   'dist_from_sample_centroid0': dist_from_sample_centroid0,
    #  9   'simp_dist_from_sample_centroid': simp_dist_from_sample_centroid,
    #  10   'ets_dist_from_sample_centroid': ets_dist_from_sample_centroid
    # })

    # static_df = pd.DataFrame({
    #     'sigma': [sigma],
    #     'rep': [rep],
    #     'simp_num_trees': [simp_num_trees],
    #     'ets_num_trees': [ets_num_trees], 
    #     'simp_num_edges': [simp_num_edges],
    #     'ets_num_edges': [ets_num_edges]
    # })

In [91]:
def error_change_proportion_by_correct_added_span_bin(results, static):
    sigma = static['sigma'].values[0]
    num_bins = 8

    correct_span = results.iloc[:, 7] - results.iloc[:, 8]

    
    results['correct_span_bin'], bin_edges = pd.qcut(
        correct_span, 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['correct_span_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    
    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
    
        bin_data = results[results['correct_span_bin'] == bin_interval]
        proportion = ((bin_data.iloc[:, 2] - bin_data.iloc[:, 3]) / bin_data.iloc[:, 2]) * 100
        
      # where 2 is simp and  3 is extended , 


        fig.add_trace(go.Histogram(
            x=proportion, 
            name='Percent Decrease',
            opacity=0.6, 
            marker_color='red',
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_layout(barmode='overlay', title=f"Sigma {sigma} Error Percent Decrease by Correct Added Span Bin", height=600)
    fig.update_yaxes(type='log')
    fig.show()
    

In [116]:
def error_by_rank(results, static):
    ranked_results = results.sort_values(by='added_span')
    fig = go.Figure()

    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]

    
    fig.add_trace(go.Scatter(x=ranked_results.iloc[:, 0], y = ranked_results.iloc[:, 3], mode='markers', name='Extended'))
# fig = px.scatter(
#     ranked_results,
#     x=ranked_results.iloc[:, 7], # Numerical Added Span
#     y='proportion',              # Your calculated error change
#     hover_name=ranked_.iloc[:, 0], # The Unique Node ID
#     color='span_bin',            # Groups nodes by bin range
#     title="Node-wise Error Change by Added Span"
# )

# Add a trend line to see if error change correlates with span length
    fig.update_layout(title=f"S{sigma} R{rep} Ranked Extended Error by Added Span", xaxis_title='Node ID', yaxis_title='Error')
 
    # fig.update_traces(marker=dict(size=8, opacity=0.7))
    fig.show()
    

In [93]:
error_change_proportion_by_time_bin(child, S0_2_R0_static)

In [85]:
def error_change_proportion_by_time_bin(results, static):
    sigma = static['sigma'].values[0]
    num_bins = 8

    #correct_span = results.iloc[:, 7] - results.iloc[:, 8]

    
    results['correct_span_bin'], bin_edges = pd.qcut(
        results.iloc[:, 1], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['correct_span_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    
    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
    
        bin_data = results[results['correct_span_bin'] == bin_interval]
        proportion = ((bin_data.iloc[:, 2] - bin_data.iloc[:, 3]) / bin_data.iloc[:, 2]) * 100
        
      # where 2 is simp and  3 is extended , 


        fig.add_trace(go.Histogram(
            x=proportion, 
            name='Percent Decrease',
            opacity=0.6, 
            marker_color='red',
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_layout(barmode='overlay', title=f"Sigma {sigma} Error Percent Decrease by Time Bin", height=600)
    fig.update_yaxes(type='log')
    fig.show()
    

In [ ]:
def error_diff_by_total_added_span_bin(results, static ):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8

    
    results['span_bin'], bin_edges = pd.qcut(
        results.iloc[:, 7], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['span_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    
    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
    
        bin_data = results[results['span_bin'] == bin_interval]
        diff = bin_data.iloc[:, 2] - bin_data.iloc[:, 3]
      


        fig.add_trace(go.Histogram(
            x=diff, 
            name='Simplified - Extended',
            opacity=0.6, 
            marker_color='red',
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_layout(barmode='overlay', title=f"S{sigma} R{rep} Error Difference by Total Added Span Bin", height=600)
    fig.show()
    

In [ ]:
def avg_error_difference_by_total_added_span_bin(results, static):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8

    results['span_bin'], bin_edges = pd.qcut(
        results.iloc[:, 7], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['span_bin'].unique())

    array = np.zeros(8)
    count = 0

    fig = go.Figure()

    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1 

        bin_data = results[results['span_bin'] == bin_interval]

        simp_errors = bin_data.iloc[:, 2]
        avg_simp_errors = np.mean(simp_errors)

        ets_errors = bin_data.iloc[:, 3]
        avg_ets_errors = np.mean(ets_errors)
        
        x = avg_simp_errors - avg_ets_errors
        array[count] = x
        count = count + 1
    
    fig.add_trace(go.Scatter(x=labels, y=array,  mode='markers'))
    fig.update_layout(title=f"S{sigma} R{rep} Average Error Difference by Total Added Span Bin", 
                     xaxis_title='Added Span Bins', yaxis_title='Average Simplified Error - Extended Error')
    
    fig.show()


In [50]:
avg_error_difference_by_wrong_added_span_bin(S0_2_R0_unary, S0_2_R0_static)

In [45]:
def error_diff_by_wrong_added_span_bin(results, static ):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8

    
    results['span_bin'], bin_edges = pd.qcut(
        results.iloc[:, 8], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['span_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)
    
    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1
    
        bin_data = results[results['span_bin'] == bin_interval]
        diff = bin_data.iloc[:, 2] - bin_data.iloc[:, 3]
      


        fig.add_trace(go.Histogram(
            x=diff, 
            name='Simplified - Extended',
            opacity=0.6, 
            marker_color='red',
            showlegend=(idx == 0)
        ), row=row, col=col)

    fig.update_layout(barmode='overlay', title=f"S{sigma} R{rep} Error Difference by Wrong Added Span Bin", height=600)
    fig.show()
    

In [48]:
def avg_error_difference_by_wrong_added_span_bin(results, static):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8

    results['span_bin'], bin_edges = pd.qcut(
        results.iloc[:, 8], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )

    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['span_bin'].unique())

    array = np.zeros(8)
    count = 0

    fig = go.Figure()

    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1 

        bin_data = results[results['span_bin'] == bin_interval]

        simp_errors = bin_data.iloc[:, 2]
        avg_simp_errors = np.mean(simp_errors)

        ets_errors = bin_data.iloc[:, 3]
        avg_ets_errors = np.mean(ets_errors)
        
        x = avg_simp_errors - avg_ets_errors
        array[count] = x
        count = count + 1
    
    fig.add_trace(go.Scatter(x=labels, y=array,  mode='markers'))
    fig.update_layout(title=f"S{sigma} R{rep} Average Error Difference by Wrong Added Span Bin", 
                     xaxis_title='Wrong Added Span Bins', yaxis_title='Average Simplified Error - Extended Error')
    
    fig.show()


In [ ]:
def num_trees_histogram(static):
    fig = go.Figure()

    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]

    fig.add_trace(go.Bar(y=static['simp_num_trees'], name='Simplified Number of Trees'))
    fig.add_trace(go.Bar(y=static['ets_num_trees'], name='Extended Number of Trees'))
    fig.update_layout(title = f"S{sigma} R{rep} Number of Trees in Simplified vs Extended", yaxis_title='Count')
    fig.show()

In [ ]:
def num_edges_histogram(static):
    fig = go.Figure()

    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]

    fig.add_trace(go.Bar(y=static['simp_num_edges'], name='Simplified Number of Edges'))
    fig.add_trace(go.Bar(y=static['ets_num_edges'], name='Extended Number of Edges'))
    fig.update_layout(title = f"S{sigma} R{rep} Number of Edges in Simplified vs Extended", yaxis_title='Count')
    fig.show()

In [ ]:
def error_by_num_children(results, nodes):
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=nodes.iloc[:, 1], y = results.iloc[:, 3], mode='markers', name='Extended'))
    fig.add_trace(go.Scatter(x=nodes.iloc[:, 1], y = results.iloc[:, 2], mode='markers', name='Simplified'))

    fig.show()


In [25]:
def avg_error_diff_by_num_children_bin(results, nodes, static):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8

    results['num_children_bin'], bin_edges = pd.qcut(
        nodes.iloc[:, 1], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )


    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['num_children_bin'].unique())

    array = np.zeros(8)
    count = 0

    fig = go.Figure()

    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1 

        bin_data = results[results['num_children_bin'] == bin_interval]

        simp_errors = bin_data.iloc[:, 2]
        avg_simp_errors = np.mean(simp_errors)

        ets_errors = bin_data.iloc[:, 3]
        avg_ets_errors = np.mean(ets_errors)
    
        x = avg_simp_errors - avg_ets_errors
        array[count] = x
        count = count + 1
    
    fig.add_trace(go.Scatter(x=labels, y=array,  mode='markers'))
    fig.update_layout(title=f"S{sigma} R{rep} Average Error Difference by Number of Children Bin", 
                     xaxis_title='Number of Children Bins', yaxis_title='Average Simplified Error - Extended Error')
    
    fig.show()


In [ ]:
def error_diff_by_num_children_bin(results, nodes, static ):
    sigma = static['sigma'].values[0]
    rep = static['rep'].values[0]
    num_bins = 8

    results['num_children_bin'], bin_edges = pd.qcut(
        nodes.iloc[:, 1], 
        q=num_bins, 
        retbins=True, 
        duplicates='drop'
    )


    labels = [f"{int(bin_edges[i])}-{int(bin_edges[i+1])}" for i in range(len(bin_edges)-1)]
    unique_bins = sorted(results['num_children_bin'].unique())

    fig = make_subplots(rows=2, cols=4, subplot_titles=labels)

    for idx, bin_interval in enumerate(unique_bins):
        row = idx // 4 + 1
        col = idx % 4 + 1

        bin_data = results[results['num_children_bin'] == bin_interval]
        diff = bin_data.iloc[:, 2] - bin_data.iloc[:, 3]



        fig.add_trace(go.Histogram(
                x=diff, 
                name='Simplified - Extended',
                opacity=0.6, 
                marker_color='red',
                showlegend=(idx == 0)
            ), row=row, col=col)

    fig.update_layout(barmode='overlay', title=f"S{sigma} R{rep} Error Difference by Number of Children Bin", height=600)
    fig.show()